In [ ]:
# Setup: run this cell first. Python standard library only; no downloads.
import math
import random
from collections import Counter
from contextlib import redirect_stdout
from io import StringIO

random.seed(23)

def check_test(name, actual, expected, tol=1e-3):
    """Report values without raising on an unfinished exercise."""
    try:
        if isinstance(expected, (int, float)) and not isinstance(expected, bool):
            matched = actual is not None and math.isclose(actual, expected, abs_tol=tol, rel_tol=0)
        else:
            matched = actual == expected
        if matched:
            print(f"✅ {name}: {actual!r}")
        else:
            print(f"❗ {name}: {actual!r} (Expected: {expected!r})")
        return bool(matched)
    except Exception as exc:
        print(f"❗ {name}: {actual!r} (Expected: {expected!r}; {type(exc).__name__}: {exc})")
        return False

def run_check(name, calculation, expected, tol=1e-3):
    """Delay evaluation so even an exception inside learner code is caught."""
    try:
        return check_test(name, calculation(), expected, tol)
    except Exception as exc:
        print(f"❗ {name}: {type(exc).__name__}: {exc} (Expected: {expected!r})")
        return False


## Intuition & Quick Walkthrough

A translation pipeline configures a target language, generates text, compares it
with a reference, and checks whether the output is safe to use. Overlap counts
are useful signals, but they cannot prove that the meaning is preserved.
The sandbox below uses a Spanish weather example, a small lookup table, and
**unigram overlap only**, not BLEU or chrF. Change the candidate and observe
how repeated or missing words affect the clipped match count.

This is the practice notebook for [Machine Translation](../docs/en.md), based on
[the reference implementation](../code/main.py). Fill each `pass`, then rerun
its cell. Red markers are expected until you implement the exercises.

The four Steps follow `Build It` in order. The reference only implements
evaluation; the call, production review, and preprocessing exercises below add
explicit offline contracts. Model calls and training are simulated, not real
translation or fine-tuning. The provider packages shown in the lesson are skipped:
**stays stdlib-first for educational clarity.** All existing reference function
signatures are preserved; new exercise functions are identified as such.


In [ ]:
# Independent sandbox: different data and settings from all exercise checks.
weather_source = "Tomorrow brings strong winds near the northern coast"
weather_language = "spa_Latn"
weather_table = {(weather_source, weather_language): "mañana soplarán vientos fuertes en la costa norte"}
weather_candidate = weather_table[(weather_source, weather_language)]
weather_reference = "mañana llegarán vientos intensos a nuestra costa norte"
weather_words = weather_candidate.split()
weather_gold = weather_reference.split()
weather_matches = sum((Counter(weather_words) & Counter(weather_gold)).values())
weather_overlap = weather_matches / len(weather_words)
print("Lookup output:", weather_candidate)
print("Token counts:", len(weather_words), len(weather_gold))
print("Clipped matches:", weather_matches)
print("Unigram overlap (not BLEU):", round(weather_overlap, 3))
print("Generation settings preview:", {"num_beams": 7, "max_new_tokens": 37, "length_penalty": 0.85})
print("Meaning still needs human review even when words overlap.")


## Step 1: a pretrained MT call

Follow the lesson's source → tokenizer → generation → decode workflow, using
its English/French sentence and generation settings. A supplied recording double
stands in for the tokenizer and model so no weights or packages are needed.
Its fixed output is a fixture, not evidence of model quality.

- `translate(src, tok, model)` — **new exercise wrapper** around the inline
  example in the docs. Tokenize with `return_tensors="pt"`, request French via
  `forced_bos_token_id`, use 5 beams, a 1.0 length penalty, and a 64-token cap;
  decode with special tokens removed and return the first string.

`src_lang="eng_Latn"` is already set on the double, just as in the docs.
The checks inspect arguments as well as output so a hardcoded translation fails.


In [ ]:
# Supplied test infrastructure, not a learner implementation.
class RecordingTokenizer:
    def __init__(self):
        self.src_lang = "eng_Latn"
        self.calls = []

    def __call__(self, text, **kwargs):
        self.calls.append(("encode", text, kwargs))
        if "text_target" in kwargs:
            return {"input_ids": [11, 12, 0], "attention_mask": [1, 1, 0], "labels": [21, 22, 0]}
        return {"input_ids": [[11, 12]], "attention_mask": [[1, 1]]}

    def convert_tokens_to_ids(self, language):
        self.calls.append(("language", language))
        return {"fra_Latn": 99}[language]

    def batch_decode(self, sequences, **kwargs):
        self.calls.append(("decode", sequences, kwargs))
        return ["Les chats courent."]

class RecordingModel:
    def __init__(self):
        self.calls = []

    def generate(self, **kwargs):
        self.calls.append(kwargs)
        return [[99, 21, 22]]

def translate(src, tok, model):
    # TODO: Pass source text through tok, model generation, and target decoding.
    # Example inputs: src="The cats are running.", supplied tok and model.
    # Example: these fixtures -> "Les chats courent."
    # Hint: Preserve the encoded input mapping; resolve the target language ID.
    # Hint: Follow the Step 1 settings and select the first decoded sequence.
    pass

step1_tok, step1_model = RecordingTokenizer(), RecordingModel()
run_check("translation", lambda: translate("The cats are running.", step1_tok, step1_model), "Les chats courent.")
run_check("tokenizer and decoding calls", lambda: step1_tok.calls, [
    ("encode", "The cats are running.", {"return_tensors": "pt"}),
    ("language", "fra_Latn"),
    ("decode", [[99, 21, 22]], {"skip_special_tokens": True}),
])
run_check("generation settings", lambda: step1_model.calls, [{
    "input_ids": [[11, 12]], "attention_mask": [[1, 1]],
    "forced_bos_token_id": 99, "num_beams": 5,
    "length_penalty": 1.0, "max_new_tokens": 64,
}])


## Step 2: BLEU and chrF

Implement the teaching metrics from `code/main.py`. Keep their exact signatures
(the reference has no type annotations). These are **not SacreBLEU-compatible
benchmark scores**: tokenization, case handling, whitespace, smoothing, and
aggregation conventions differ. The reference computes sentence-level scores.

- `tokenize(text)` — lowercase, separate periods and commas, then split whitespace.
- `ngrams(tokens, n)` — contiguous tuples: $g_i=(t_i,\ldots,t_{i+n-1})$,
  for $0\le i\le L-n$, giving $\max(0,L-n+1)$ tuples. Assume positive integer $n$.
- `ngram_precision(hyp_tokens, ref_tokens, n)` — clipped precision
  $p_n=\frac{\sum_g\min(C_h(g),C_r(g))}{\sum_g C_h(g)}$; zero if the denominator is zero.
- `brevity_penalty(hyp_len, ref_len)` — for hypothesis length $c$ and reference
  length $r$, $BP=1$ if $c\ge r$; otherwise $BP=0$ if $c=0$;
  otherwise $BP=\exp(1-r/c)$. Branch order preserves the reference's $BP(0,0)=1$.
- `simple_bleu(hypothesis, reference, max_n=4)` —
  $100\,BP\exp(\frac{1}{N}\sum_{n=1}^{N}\log p_n)$, or zero if any $p_n=0$.
- `chrf(hypothesis, reference, n=6, beta=2)` — lowercase both strings and retain
  spaces/punctuation. For each usable order $k$, calculate clipped character
  matches $m_k$, $P_k=m_k/H_k$, and $R_k=m_k/T_k$. Skip orders with $H_k=0$ or
  $T_k=0$. Average over usable orders $K$: $P=\frac1{|K|}\sum P_k$,
  $R=\frac1{|K|}\sum R_k$. Return
  $100\frac{(1+\beta^2)PR}{\beta^2P+R}$, or zero if there are no usable orders or $P+R=0$.
  Its nested `char_ngrams(text, k)` returns windows $s[i:i+k]$ (strings, not tuples).

Work through the helpers first, then combine them. The main check uses the
exact French sentence from the docs; extra checks expose clipping, short-text,
empty-text, and partial-overlap behavior. No smoothing is applied to BLEU.


In [ ]:
def tokenize(text):
    # TODO: Convert text into lowercase word/punctuation tokens.
    # Example inputs: text="Les chats courent."
    # Example: -> ["les", "chats", "courent", "."]
    # Hint: Separate periods and commas before whitespace splitting; empty text has no tokens.
    pass

def ngrams(tokens, n):
    # TODO: Collect every contiguous n-token tuple from tokens; n is positive.
    # Example inputs: tokens=["les", "chats", "courent", "."], n=2
    # Example: -> [("les", "chats"), ("chats", "courent"), ("courent", ".")]
    # Hint: Advance a window one position at a time; a short sequence gives no windows.
    pass

def ngram_precision(hyp_tokens, ref_tokens, n):
    # TODO: Measure hypothesis n-gram precision with reference-clipped counts.
    # Example inputs: hyp_tokens=["les", "les", "chats"], ref_tokens=["les", "chats"], n=1
    # Example: -> 0.6666666667
    # Hint: Count both multisets, sum minimum counts, divide by hypothesis count; handle zero total.
    pass

def brevity_penalty(hyp_len, ref_len):
    # TODO: Compute BP from hypothesis length hyp_len and reference length ref_len.
    # Example inputs: hyp_len=2, ref_len=4
    # Example: -> 0.3678794412
    # Hint: Test c >= r first, then empty c, then exp(1-r/c); use nonnegative lengths.
    pass

def simple_bleu(hypothesis, reference, max_n=4):
    # TODO: Combine clipped precisions through order max_n with BP on token lengths.
    # Example inputs: hypothesis="Les chats courent.", reference="Les chats courent."
    # Example: -> 100.0; hypothesis="Les" with the same reference -> 0.0
    # Hint: Tokenize, gather precisions, reject zero before logs, then scale the geometric mean by BP and 100.
    pass

def chrf(hypothesis, reference, n=6, beta=2):
    # TODO: Average character precisions/recalls across usable orders, then compute F-beta on a 100 scale.
    # Example inputs: hypothesis="Les chats courent.", reference="Les chats courent."
    # Example: -> 100.0; empty hypothesis -> 0.0
    # Hint: Lowercase while keeping whitespace, clip counts per order, skip empty denominators.
    # Hint: Use the formula in the brief; handle no usable orders and P+R=0.
    def char_ngrams(text, k):
        # TODO: Return all contiguous k-character strings from text.
        # Example inputs: text="chats", k=2
        # Example: -> ["ch", "ha", "at", "ts"]
        # Hint: Slide a fixed-width window; return no windows when text is shorter than k.
        pass
    pass

run_check("tokenize docs sentence", lambda: tokenize("Les chats courent."), ["les", "chats", "courent", "."])
run_check("comma and whitespace", lambda: tokenize(" Les, CHATS. "), ["les", ",", "chats", "."])
run_check("bigrams", lambda: ngrams(["les", "chats", "courent", "."], 2), [("les", "chats"), ("chats", "courent"), ("courent", ".")])
run_check("short ngrams", lambda: ngrams(["les"], 4), [])
run_check("clipped precision", lambda: ngram_precision(["les", "les", "chats"], ["les", "chats"], 1), 2/3)
run_check("empty precision", lambda: ngram_precision([], ["les"], 1), 0.0)
run_check("short BP", lambda: brevity_penalty(2, 4), math.exp(-1))
run_check("long BP", lambda: brevity_penalty(6, 4), 1.0)
run_check("empty BP", lambda: brevity_penalty(0, 4), 0.0)
run_check("both empty BP (reference convention)", lambda: brevity_penalty(0, 0), 1.0)
run_check("BLEU exact docs example", lambda: simple_bleu("Les chats courent.", "Les chats courent."), 100.0)
run_check("BLEU short, no smoothing", lambda: simple_bleu("Les", "Les chats courent."), 0.0)
run_check("BLEU partial unigram", lambda: simple_bleu("Les chiens mangent.", "Les chats courent.", max_n=1), 50.0)
run_check("chrF exact docs example", lambda: chrf("Les chats courent.", "Les chats courent."), 100.0)
run_check("chrF partial reference case", lambda: chrf("Les chiens mangent.", "Les chats courent."), 34.7947051048)
run_check("chrF empty", lambda: chrf("", "Les chats courent."), 0.0)
run_check("chrF short / skip absent orders", lambda: chrf("Les", "Les"), 100.0)
run_check("chrF disjoint", lambda: chrf("xxx", "Les"), 0.0)


## Step 3: what breaks in production

Turn the five failure modes from the docs into a deterministic review exercise.
No function for this exists in the reference; implement this **new exercise**:

- `review_translation(source, candidate, detected_language, expected_language, glossary, formal, unsupported_content)`
  returns an ordered list of issue labels: `hallucination`, `off-target`,
  `terminology-drift`, `formality-mismatch`, `length-explosion`.

`unsupported_content` is a human-provided judgment, not an automatic semantic
detector. `detected_language` is an externally supplied language-ID result;
`None` counts as an off-target/unknown result requiring review. For each glossary
entry whose source phrase occurs (case-insensitive substring), require its
target phrase in the candidate. For this French-only register exercise, formal
text containing the whitespace-delimited token `tu` is flagged. This deliberately
limited rule does not identify every register mismatch.

Flag length explosion when target whitespace-token count exceeds
$\max(4, 3s)$, where $s$ is the source whitespace-token count. This is an exercise
threshold, not a universal translation-quality criterion. Passing these checks
does not establish semantic accuracy or grant permission to ship.


In [ ]:
def review_translation(source, candidate, detected_language, expected_language, glossary, formal, unsupported_content):
    # TODO: Return applicable issue labels in the order specified in the brief.
    # Example inputs: source="The cats are running.", candidate="Les chats courent.",
    # detected_language="fra_Latn", expected_language="fra_Latn", glossary={}, formal=True, unsupported_content=False.
    # Example: -> []; detected_language=None instead -> ["off-target"].
    # Hint: Evaluate independent checks; do not stop at the first issue.
    # Hint: Use case-insensitive glossary matching and whitespace tokens for length/register.
    # Hint: The supplied language and content judgments are inputs, not facts inferred from overlap scores.
    pass

run_check("docs translation review", lambda: review_translation("The cats are running.", "Les chats courent.", "fra_Latn", "fra_Latn", {}, True, False), [])
run_check("missing language check", lambda: review_translation("The cats are running.", "Les chats courent.", None, "fra_Latn", {}, True, False), ["off-target"])
run_check("docs terminology drift", lambda: review_translation("Sign up", "créer un compte", "fra_Latn", "fra_Latn", {"Sign up": "s'inscrire"}, False, False), ["terminology-drift"])
run_check("formal register", lambda: review_translation("Sign up", "tu peux t'inscrire", "fra_Latn", "fra_Latn", {}, True, False), ["formality-mismatch"])
run_check("unsupported content judgment", lambda: review_translation("The cats are running.", "Les chats courent.", "fra_Latn", "fra_Latn", {}, False, True), ["hallucination"])
run_check("short-input explosion", lambda: review_translation("Sign up", "Veuillez créer un compte pour continuer à utiliser tous nos services", "fra_Latn", "fra_Latn", {}, True, False), ["length-explosion"])
run_check("combined issues", lambda: review_translation("Sign up", "tu can register and receive a prize today", "eng_Latn", "fra_Latn", {"Sign up": "s'inscrire"}, True, True), ["hallucination", "off-target", "terminology-drift", "formality-mismatch", "length-explosion"])


## Step 4: fine-tuning for a domain

Use the exact legal-domain pair and preprocessing settings from the docs.
This offline exercise verifies data plumbing and training configuration. It does
not train a model or measure translation improvement.

- `preprocess(ex)` — exact signature from the docs. Pass `ex["src"]` and
  `ex["tgt"]` to the global tokenizer `tok`, with truncation, a maximum length
  of 128, and `padding="max_length"`. Return the tokenizer mapping.
- `training_config()` — **new exercise wrapper**: return a dict containing the
  docs' four training settings: `output_dir="out"`,
  `per_device_train_batch_size=4`, `num_train_epochs=3`, `learning_rate=3e-5`.

The fixture uses tiny sentinel token IDs to inspect the call; it does not produce
a real padded 128-token tensor. We reproduce the docs' preprocessing contract.
For actual training, padding in labels also needs appropriate loss masking, and
quality must be measured on held-out domain examples.


In [ ]:
tok = RecordingTokenizer()

def preprocess(ex):
    # TODO: Encode one parallel example using global tok and the docs' target/padding settings.
    # Example inputs: ex={"src": "The defendant pleaded guilty.", "tgt": "L'accusé a plaidé coupable."}
    # Example: supplied fixture -> {"input_ids": [11, 12, 0], "attention_mask": [1, 1, 0], "labels": [21, 22, 0]}
    # Hint: Source and target have distinct roles; forward both to the tokenizer, then preserve its mapping.
    pass

def training_config():
    # TODO: Package the four training settings from Step 4 into a dictionary.
    # Example inputs: none.
    # Example: result["num_train_epochs"] -> 3; result["learning_rate"] -> 0.00003.
    # Hint: Keep output directory, batch size, epoch count, and learning rate under the documented keys.
    pass

legal_example = {"src": "The defendant pleaded guilty.", "tgt": "L'accusé a plaidé coupable."}
run_check("preprocess legal pair", lambda: preprocess(legal_example), {"input_ids": [11, 12, 0], "attention_mask": [1, 1, 0], "labels": [21, 22, 0]})
run_check("preprocessing arguments", lambda: tok.calls, [("encode", "The defendant pleaded guilty.", {
    "text_target": "L'accusé a plaidé coupable.", "truncation": True,
    "max_length": 128, "padding": "max_length",
})])
run_check("training settings", training_config, {"output_dir": "out", "per_device_train_batch_size": 4, "num_train_epochs": 3, "learning_rate": 3e-5})


## Extra 1: explain the unsmoothed BLEU caveat

The reference's `simple_bleu_note()` is not a separate Build It step. Implement
it with its exact signature, returning the reference's explanatory string below.
This is a documentation exercise: explain why a missing higher-order match can
erase an otherwise useful partial score.

- `simple_bleu_note()` — return the supplied note. The relevant property is
  $p_k=0 \Rightarrow \mathrm{BLEU}=0$ for this unsmoothed implementation.


In [ ]:
def simple_bleu_note():
    # TODO: Return the reference note about zero precisions and smoothing, exactly as shown below.
    # Example inputs: none.
    # Example: return text beginning "simple_bleu above has no smoothing:".
    # Hint: Join readable text fragments without accidentally dropping spaces.
    pass

expected_note = (
    "simple_bleu above has no smoothing: one zero-precision n-gram drops "
    "the score to 0.0. This punishes short hypotheses; production uses "
    "epsilon / NIST / add-k smoothing via sacrebleu."
)
run_check("BLEU explanatory note", simple_bleu_note, expected_note)


## Extra 2: assemble the evaluation report

The reference's `main()` is a runnable reporting entry point outside the docs'
four steps. Complete it after the metric functions and the note work.

- `main()` — print the reference's four hypothesis/reference pairs with BLEU and
  chrF to one decimal place, followed by the note and the two supplied closing
  lines. Return `None`, matching the reference.

Match the reference column widths: hypothesis 40, reference 25, metric headings
and values 6, separated by two spaces. Do not hardcode computed metric scores.
The report's claim about a one-point BLEU difference is copied for reference
parity; meaningful differences depend on the test set and statistical uncertainty.


In [ ]:
def main():
    # TODO: Print the four-case metric table and explanatory footer from the reference.
    # Example inputs: none; use the four REPORT_CASES below.
    # Example: the exact-match row displays BLEU 100.0 and chrF 100.0.
    # Hint: Compute both scores for each pair, format one decimal, then print a blank line and footer.
    # Hint: Leave the return value implicit; reporting uses stdout.
    pass

REPORT_CASES = [
    ("Les chats courent.", "Les chats courent."),
    ("Les chats sont en train de courir.", "Les chats courent."),
    ("Les chiens mangent.", "Les chats courent."),
    ("Les", "Les chats courent."),
]
REPORT_FOOTER = [
    expected_note,
    "BLEU under 1 point is noise. chrF catches morphological partials BLEU misses.",
    "For real work, use sacrebleu (pip install sacrebleu) instead of this teaching version.",
]

# Expected scores are fixed oracle values, not calls to learner functions.
expected_scores = [(100.0, 100.0), (0.0, 56.2800121946), (0.0, 34.7947051048), (0.0, 14.0446650124)]
expected_report = f"{'hypothesis':40s}  {'reference':25s}  {'BLEU':>6}  {'chrF':>6}\n"
for (hyp, ref), (bleu_score, chrf_score) in zip(REPORT_CASES, expected_scores):
    expected_report += f"{hyp:40s}  {ref:25s}  {bleu_score:6.1f}  {chrf_score:6.1f}\n"
expected_report += "\n" + "\n".join(REPORT_FOOTER) + "\n"

try:
    report_buffer = StringIO()
    with redirect_stdout(report_buffer):
        report_return = main()
    check_test("main report text", report_buffer.getvalue(), expected_report)
    check_test("main return value (also requires report)", (report_return, bool(report_buffer.getvalue())), (None, True))
except Exception as exc:
    print(f"❗ main report: {type(exc).__name__}: {exc} (Expected: four scored rows and footer)")
